In [ ]:
!unzip archive.zip

Archive:  archive.zip
  inflating: Cars in Movies.csv      


In [ ]:
import pandas as pd

df = pd.read_csv('Cars in Movies.csv', sep=';')

In [ ]:
df.shape

(1756636, 21)

In [ ]:
df.columns

Index(['Car Id', 'Car Full Name', 'Car Name', 'Car Year', 'Brand',
       'Movie Title', 'Movie Type', 'Movie Years and Episodes', 'IMDb Link',
       'Class', 'Car Type', 'Car Origin', 'Car Built In', 'Car Made For',
       'Time In Movie', 'Car Stars', 'Movie Eng Title', 'Movie Original Title',
       'Car Image Link', 'Movie Year', 'IMDb Movie Id'],
      dtype='object')

In [ ]:
df['Car Name'].nunique()

172297

In [ ]:
# translate all elements to English before matching

In [ ]:
df['Car Name'].sample(10)

,Car Name
606604,Volkswagen Golf GL I
611206,Nissan Frontier
849472,Opel Rekord Ascona
1624135,Hyundai Avante
907572,Proton Waja
1271158,Toyota Prius III
916697,Honda Civic
162274,Oldsmobile Ninety-Eight Regency
846405,Simca 1000 Coupé
990838,Chevrolet Step-Van


In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

def get_matching_score(text1, text2):
    embeddings = model.encode([text1, text2])
    similarity = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return float(similarity)

# Example
score = get_matching_score("Toyota Camry 2020", "Toyota Camry 2020")
# Returns ~1.0 for perfect match

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
score

0.9999999403953552

In [ ]:
from fuzzywuzzy import fuzz
import re

def normalize_vehicle_text(text):
    # Remove extra spaces, special characters, standardize case
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text.lower())
    return ' '.join(text.split())

def fuzzy_match_score(text1, text2):
    normalized1 = normalize_vehicle_text(text1)
    normalized2 = normalize_vehicle_text(text2)

    # Combine multiple fuzzy matching techniques
    ratio = fuzz.ratio(normalized1, normalized2)
    partial_ratio = fuzz.partial_ratio(normalized1, normalized2)
    token_sort = fuzz.token_sort_ratio(normalized1, normalized2)

    return (ratio + partial_ratio + token_sort) / 300  # Normalize to 0-1

In [ ]:
import re

def rule_based_vehicle_match(vehicle1, vehicle2):
    score = 0.0
    max_score = 1.0

    # Extract components (simplified - you might want more sophisticated parsing)
    def extract_components(vehicle_str):
        # This is simplified - consider using proper vehicle parsing libraries
        words = vehicle_str.lower().split()
        return {
            'brand': words[0] if words else '',
            'model': ' '.join(words[1:]) if len(words) > 1 else ''
        }

    comp1 = extract_components(vehicle1)
    comp2 = extract_components(vehicle2)

    # Brand matching (highest weight)
    if comp1['brand'] and comp2['brand']:
        if comp1['brand'] == comp2['brand']:
            score += 0.6  # Brand match is crucial
        elif comp1['brand'] in comp2['brand'] or comp2['brand'] in comp1['brand']:
            score += 0.4  # Partial brand match

    # Model matching
    if comp1['model'] and comp2['model']:
        model_similarity = fuzz.ratio(comp1['model'], comp2['model']) / 100
        score += model_similarity * 0.4

    return min(score, max_score)

In [ ]:
def hybrid_matching_score(vehicle1, vehicle2, weights=None):
    if weights is None:
        weights = {'fuzzy': 0.3, 'embedding': 0.4, 'rules': 0.3}

    # Multiple scoring methods
    fuzzy_score = fuzzy_match_score(vehicle1, vehicle2)
    embedding_score = get_matching_score(vehicle1, vehicle2)
    rule_score = rule_based_vehicle_match(vehicle1, vehicle2)

    # Weighted combination
    final_score = (weights['fuzzy'] * fuzzy_score +
                   weights['embedding'] * embedding_score +
                   weights['rules'] * rule_score)

    return {
        'final_score': final_score,
        'components': {
            'fuzzy': fuzzy_score,
            'embedding': embedding_score,
            'rules': rule_score
        }
    }

In [ ]:
# For more sophisticated matching, use models trained on similar tasks
from sentence_transformers import CrossEncoder

# Use a model trained on semantic textual similarity
model = CrossEncoder('cross-encoder/stsb-roberta-base')

def cross_encoder_score(text1, text2):
    scores = model.predict([[text1, text2]])
    return float(scores[0])